Get the Dataset from here: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce?resource=download

My Directory Structure: 

##### aida-g2-retail-data-platform

##### ├── olist_dataset/
##### └── notebooks/DataPrep&Cleaning.ipynb

In [1]:
from pathlib import Path

folder_path = Path("../olist_dataset")

# Get and sort all CSV filenames in the folder
csv_files = sorted([file.name for file in folder_path.glob("*.csv")])

for idx, file_name in enumerate(csv_files, start=1):
    print(f"{idx}. {file_name}")

1. olist_customers_dataset.csv
2. olist_geolocation_dataset.csv
3. olist_order_items_dataset.csv
4. olist_order_payments_dataset.csv
5. olist_order_reviews_dataset.csv
6. olist_orders_dataset.csv
7. olist_products_dataset.csv
8. olist_sellers_dataset.csv
9. product_category_name_translation.csv


In [3]:
import pandas as pd
from pathlib import Path

def validate_olist_dataset(data_dir: str = "../olist_dataset"):
    path = Path(data_dir)
    print("Loading datasets for validation...\n")
    
    # Load all tables
    orders = pd.read_csv(path / "olist_orders_dataset.csv")
    items = pd.read_csv(path / "olist_order_items_dataset.csv")
    payments = pd.read_csv(path / "olist_order_payments_dataset.csv")
    reviews = pd.read_csv(path / "olist_order_reviews_dataset.csv")
    customers = pd.read_csv(path / "olist_customers_dataset.csv")
    products = pd.read_csv(path / "olist_products_dataset.csv")
    sellers = pd.read_csv(path / "olist_sellers_dataset.csv")
    translations = pd.read_csv(path / "product_category_name_translation.csv")
    geo = pd.read_csv(path / "olist_geolocation_dataset.csv")

    results = []

    def log_check(category: str, check_name: str, passed: bool, details: str):
        status = "PASS" if passed else "WARN/FAIL"
        results.append({
            "Category": category,
            "Check": check_name,
            "Status": status,
            "Details": details
        })

    # 1. PRIMARY KEY UNIQUENESS & NULL CHECKS
    log_check(
        "Primary Keys", "Orders: unique & non-null order_id",
        orders["order_id"].notna().all() and orders["order_id"].is_unique,
        f"Total: {len(orders)}, Nulls: {orders['order_id'].isna().sum()}, Duplicates: {orders['order_id'].duplicated().sum()}"
    )
    log_check(
        "Primary Keys", "Customers: unique & non-null customer_id",
        customers["customer_id"].notna().all() and customers["customer_id"].is_unique,
        f"Total: {len(customers)}, Unique people (customer_unique_id): {customers['customer_unique_id'].nunique()}"
    )
    log_check(
        "Primary Keys", "Products: unique & non-null product_id",
        products["product_id"].notna().all() and products["product_id"].is_unique,
        f"Total: {len(products)}, Nulls: {products['product_id'].isna().sum()}"
    )
    log_check(
        "Primary Keys", "Sellers: unique & non-null seller_id",
        sellers["seller_id"].notna().all() and sellers["seller_id"].is_unique,
        f"Total: {len(sellers)}, Nulls: {sellers['seller_id'].isna().sum()}"
    )
    log_check(
        "Primary Keys", "Reviews: unique review_id",
        reviews["review_id"].is_unique,
        f"Total rows: {len(reviews)}, Duplicate review_ids: {reviews['review_id'].duplicated().sum()}"
    )

    # 2. REFERENTIAL INTEGRITY (FOREIGN KEYS)
    missing_cust = set(orders["customer_id"]) - set(customers["customer_id"])
    log_check(
        "Foreign Keys", "Orders -> Customers (customer_id)",
        len(missing_cust) == 0,
        f"Orphaned customer_ids in orders: {len(missing_cust)}"
    )

    orders_without_items = set(orders["order_id"]) - set(items["order_id"])
    log_check(
        "Foreign Keys", "Orders -> Order Items (every order has >=1 item)",
        len(orders_without_items) == 0,
        f"Orders with 0 items: {len(orders_without_items)} (usually cancelled/unavailable orders)"
    )

    orders_without_payments = set(orders["order_id"]) - set(payments["order_id"])
    log_check(
        "Foreign Keys", "Orders -> Order Payments (every order has >=1 payment)",
        len(orders_without_payments) == 0,
        f"Orders with 0 payment records: {len(orders_without_payments)}"
    )

    missing_products = set(items["product_id"]) - set(products["product_id"])
    log_check(
        "Foreign Keys", "Order Items -> Products (product_id)",
        len(missing_products) == 0,
        f"Orphaned product_ids: {len(missing_products)}"
    )

    missing_sellers = set(items["seller_id"]) - set(sellers["seller_id"])
    log_check(
        "Foreign Keys", "Order Items -> Sellers (seller_id)",
        len(missing_sellers) == 0,
        f"Orphaned seller_ids: {len(missing_sellers)}"
    )

    untranslated_cats = set(products["product_category_name"].dropna()) - set(translations["product_category_name"])
    log_check(
        "Foreign Keys", "Products -> Category Translations",
        len(untranslated_cats) == 0,
        f"Categories missing English translation: {untranslated_cats if untranslated_cats else 0}"
    )

    # 3. DOMAIN & VALUE LOGIC CHECKS
    neg_prices = (items["price"] <= 0).sum()
    neg_freight = (items["freight_value"] < 0).sum()
    log_check(
        "Business Logic", "Item prices > 0 & freight >= 0",
        neg_prices == 0 and neg_freight == 0,
        f"Price <= 0: {neg_prices}, Freight < 0: {neg_freight}"
    )

    zero_payments = (payments["payment_value"] <= 0).sum()
    log_check(
        "Business Logic", "Payment values > 0",
        zero_payments == 0,
        f"Payments <= 0: {zero_payments} (vouchers/zero-value edge cases)"
    )

    invalid_scores = (~reviews["review_score"].isin([1, 2, 3, 4, 5])).sum()
    log_check(
        "Business Logic", "Review scores strictly between 1 and 5",
        invalid_scores == 0,
        f"Invalid review scores: {invalid_scores}"
    )

    # 4. TEMPORAL / CHRONOLOGICAL CONSISTENCY
    ts_cols = [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
    for col in ts_cols:
        orders[col] = pd.to_datetime(orders[col])

    # Purchase should happen before or at approval
    bad_approval = (orders["order_approved_at"] < orders["order_purchase_timestamp"]).sum()
    # Carrier handoff should happen before or at customer delivery
    bad_delivery = (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]).sum()
    log_check(
        "Timestamps", "Chronological order lifecycle (Purchase <= Approved, Carrier <= Delivered)",
        bad_approval == 0 and bad_delivery == 0,
        f"Approved before purchase: {bad_approval} | Delivered before carrier handoff: {bad_delivery}"
    )

    # 5. FINANCIAL RECONCILIATION (Items + Freight vs. Total Payment)
    item_totals = items.groupby("order_id").apply(lambda df: (df["price"] + df["freight_value"]).sum()).rename("expected_total")
    pay_totals = payments.groupby("order_id")["payment_value"].sum().rename("paid_total")
    recon = pd.concat([item_totals, pay_totals], axis=1).dropna()
    mismatches = (recon["expected_total"] - recon["paid_total"]).abs() > 0.05  # 5-cent rounding tolerance
    log_check(
        "Financials", "Order total (price + freight) matches sum of payments",
        mismatches.sum() == 0,
        f"Orders with >$0.05 payment discrepancy: {mismatches.sum()} out of {len(recon)}"
    )

    # 6. GEOLOCATION DUPLICATE CHECK
    dup_zips = geo["geolocation_zip_code_prefix"].duplicated().sum()
    log_check(
        "Dimensions", "Geolocation zip code uniqueness",
        dup_zips == 0,
        f"Total rows: {len(geo)}, Duplicate zip prefixes: {dup_zips} (must aggregate before joining!)"
    )

    # Display Report
    report_df = pd.DataFrame(results)
    print(report_df.to_string(index=False))
    return report_df

if __name__ == "__main__":
    validate_olist_dataset("../olist_dataset")

Loading datasets for validation...



C:\Users\jaivi\AppData\Local\Temp\ipykernel_26836\833117312.py:144: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  item_totals = items.groupby("order_id").apply(lambda df: (df["price"] + df["freight_value"]).sum()).rename("expected_total")


      Category                                                                      Check    Status                                                                                               Details
  Primary Keys                                         Orders: unique & non-null order_id      PASS                                                                 Total: 99441, Nulls: 0, Duplicates: 0
  Primary Keys                                   Customers: unique & non-null customer_id      PASS                                               Total: 99441, Unique people (customer_unique_id): 96096
  Primary Keys                                     Products: unique & non-null product_id      PASS                                                                                Total: 32951, Nulls: 0
  Primary Keys                                       Sellers: unique & non-null seller_id      PASS                                                                                 Total: 3095,

In [4]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime, timedelta

class OlistAPISimulator:
    def __init__(self, data_dir: str = "../olist_dataset"):
        self.data_dir = Path(data_dir)
        self._load_datasets()

    def _load_datasets(self):
        """Loads raw CSVs into memory to act as the mock backend database."""
        self.orders = pd.read_csv(
            self.data_dir / "olist_orders_dataset.csv",
            parse_dates=["order_purchase_timestamp"]
        )
        self.items = pd.read_csv(self.data_dir / "olist_order_items_dataset.csv")
        self.payments = pd.read_csv(self.data_dir / "olist_order_payments_dataset.csv")
        self.reviews = pd.read_csv(self.data_dir / "olist_order_reviews_dataset.csv")
        self.customers = pd.read_csv(self.data_dir / "olist_customers_dataset.csv")
        self.products = pd.read_csv(self.data_dir / "olist_products_dataset.csv")
        self.sellers = pd.read_csv(self.data_dir / "olist_sellers_dataset.csv")
        self.translations = pd.read_csv(self.data_dir / "product_category_name_translation.csv")
        self.geolocation = pd.read_csv(self.data_dir / "olist_geolocation_dataset.csv")

    def get_orders_batch(self, target_date: str, window_days: int = 1) -> list[dict]:
        """
        Simulates GET /api/v1/orders?start_date=YYYY-MM-DD&days=1
        Returns nested JSON records just like a real e-commerce REST API.
        """
        start_dt = pd.to_datetime(target_date)
        end_dt = start_dt + timedelta(days=window_days)

        # Filter orders within the requested time window
        batch_orders = self.orders[
            (self.orders["order_purchase_timestamp"] >= start_dt) &
            (self.orders["order_purchase_timestamp"] < end_dt)
        ].copy()

        if batch_orders.empty:
            return []

        order_ids = batch_orders["order_id"].tolist()

        # Pre-filter child tables for fast lookup
        batch_items = self.items[self.items["order_id"].isin(order_ids)]
        batch_payments = self.payments[self.payments["order_id"].isin(order_ids)]
        batch_reviews = self.reviews[self.reviews["order_id"].isin(order_ids)]
        batch_customers = self.customers[self.customers["customer_id"].isin(batch_orders["customer_id"])]

        # Build nested API response payload
        api_response = []
        for _, order in batch_orders.iterrows():
            oid = order["order_id"]
            cid = order["customer_id"]

            cust_record = batch_customers[batch_customers["customer_id"] == cid].to_dict(orient="records")
            
            payload = {
                "order_id": oid,
                "order_status": order["order_status"],
                "timestamps": {
                    "purchased_at": str(order["order_purchase_timestamp"]),
                    "approved_at": str(order["order_approved_at"]),
                    "delivered_carrier_at": str(order["order_delivered_carrier_date"]),
                    "delivered_customer_at": str(order["order_delivered_customer_date"]),
                    "estimated_delivery_at": str(order["order_estimated_delivery_date"]),
                },
                "customer": cust_record[0] if cust_record else None,
                "items": batch_items[batch_items["order_id"] == oid].drop(columns=["order_id"]).to_dict(orient="records"),
                "payments": batch_payments[batch_payments["order_id"] == oid].drop(columns=["order_id"]).to_dict(orient="records"),
                "reviews": batch_reviews[batch_reviews["order_id"] == oid].drop(columns=["order_id"]).to_dict(orient="records"),
            }
            api_response.append(payload)

        return api_response

    def get_catalog_reference(self) -> dict:
        """Simulates GET /api/v1/catalog (dimension lookup tables)."""
        products_enriched = self.products.merge(
            self.translations, on="product_category_name", how="left"
        )
        return {
            "products": products_enriched.to_dict(orient="records"),
            "sellers": self.sellers.to_dict(orient="records"),
        }


# --- Example Usage: Simulating a Daily Ingestion Run ---
if __name__ == "__main__":
    api = OlistAPISimulator(data_dir="../olist_dataset")

    # Simulate scraping orders for November 24, 2017 (Black Friday spike in Olist!)
    simulated_date = "2017-11-24"
    orders_payload = api.get_orders_batch(target_date=simulated_date, window_days=1)

    print(f"Fetched {len(orders_payload)} orders for {simulated_date}")
    if orders_payload:
        print("\nSample JSON Payload (First Order):")
        print(json.dumps(orders_payload[0], indent=2, default=str))

Fetched 1176 orders for 2017-11-24

Sample JSON Payload (First Order):
{
  "order_id": "6ea2f835b4556291ffdc53fa0b3b95e8",
  "order_status": "delivered",
  "timestamps": {
    "purchased_at": "2017-11-24 21:27:48",
    "approved_at": "2017-11-25 00:21:09",
    "delivered_carrier_at": "2017-12-13 21:14:05",
    "delivered_customer_at": "2017-12-28 18:59:23",
    "estimated_delivery_at": "2017-12-21 00:00:00"
  },
  "customer": {
    "customer_id": "c7340080e394356141681bd4c9b8fe31",
    "customer_unique_id": "3e4fd73f1e86b135b9b121d6abbe9597",
    "customer_zip_code_prefix": 19400,
    "customer_city": "presidente venceslau",
    "customer_state": "SP"
  },
  "items": [
    {
      "order_item_id": 1,
      "product_id": "be021417a6acb56b9b50d3fd2714baa8",
      "seller_id": "f5f46307a4d15880ca14fab4ad9dfc9b",
      "shipping_limit_date": "2017-11-30 00:21:09",
      "price": 339.0,
      "freight_value": 17.12
    }
  ],
  "payments": [
    {
      "payment_sequential": 1,
      "payme